<a href="https://colab.research.google.com/github/Hujjathullah-S-T/Chromatic-9-Graphs/blob/master/BOOST_Isomorphism_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y libboost-graph-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libboost-graph1.74-dev libboost-graph1.74.0 libboost-regex1.74-dev
  libboost-regex1.74.0 libboost-serialization1.74-dev
  libboost-serialization1.74.0 libboost-test1.74-dev libboost-test1.74.0
The following NEW packages will be installed:
  libboost-graph-dev libboost-graph1.74-dev libboost-graph1.74.0
  libboost-regex1.74-dev libboost-regex1.74.0 libboost-serialization1.74-dev
  libboost-serialization1.74.0 libboost-test1.74-dev libboost-test1.74.0
0 upgraded, 9 newly installed, 0 to remove and 26 not upgraded.
Need to get 3,461 kB of archives.
After this operation, 29.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubunt

In [2]:
%%writefile isomorphism_check.cpp

#include <boost/graph/adjacency_list.hpp>
#include <boost/graph/vf2_sub_graph_iso.hpp>

#include <iostream>
#include <fstream>
#include <sstream>
#include <string>
#include <vector>
#include <utility>
#include <iomanip>
#include <algorithm>

using namespace std;

// ------------------------------------------------------------
// Graph type
// ------------------------------------------------------------
using Graph = boost::adjacency_list<
    boost::vecS,
    boost::vecS,
    boost::undirectedS
>;

// ------------------------------------------------------------
// Structure for one graph from CSV
// ------------------------------------------------------------
struct GraphRecord {
    long long generated_id;
    int n_nodes;
    int n_edges;
    int chromatic_number;

    string edge_list;
    string adjacency_upper_triangular;

    Graph graph;
};

// ------------------------------------------------------------
// Split CSV line
// The uploaded CSV files do not contain quoted commas inside
// fields, so a simple CSV split is sufficient.
// ------------------------------------------------------------
vector<string> split_csv(const string& line) {

    vector<string> fields;
    string field;
    stringstream ss(line);

    while (getline(ss, field, ',')) {
        fields.push_back(field);
    }

    return fields;
}

// ------------------------------------------------------------
// Remove surrounding quotation marks if present
// ------------------------------------------------------------
string remove_quotes(string s) {

    if (s.size() >= 2 &&
        s.front() == '"' &&
        s.back() == '"') {

        s = s.substr(1, s.size() - 2);
    }

    return s;
}

// ------------------------------------------------------------
// Build graph from edge_list
//
// Example:
// 0-1;0-2;1-3;2-3
// ------------------------------------------------------------
Graph build_graph_from_edge_list(
    int n_nodes,
    const string& edge_list
) {

    Graph g(n_nodes);

    if (edge_list.empty())
        return g;

    stringstream ss(edge_list);
    string edge;

    while (getline(ss, edge, ';')) {

        if (edge.empty())
            continue;

        size_t dash = edge.find('-');

        if (dash == string::npos)
            continue;

        int u = stoi(edge.substr(0, dash));
        int v = stoi(edge.substr(dash + 1));

        if (u >= 0 && u < n_nodes &&
            v >= 0 && v < n_nodes &&
            u != v) {

            add_edge(u, v, g);
        }
    }

    return g;
}

// ------------------------------------------------------------
// Read CSV
// ------------------------------------------------------------
vector<GraphRecord> read_csv(const string& filename) {

    vector<GraphRecord> graphs;

    ifstream file(filename);

    if (!file.is_open()) {
        cerr << "ERROR: Cannot open file: "
             << filename << endl;
        exit(1);
    }

    string line;

    // Header
    getline(file, line);

    while (getline(file, line)) {

        if (line.empty())
            continue;

        vector<string> fields = split_csv(line);

        if (fields.size() < 6) {
            cerr << "WARNING: Invalid CSV row. Skipping.\n";
            continue;
        }

        GraphRecord record;

        try {

            record.generated_id =
                stoll(remove_quotes(fields[0]));

            record.n_nodes =
                stoi(remove_quotes(fields[1]));

            record.n_edges =
                stoi(remove_quotes(fields[2]));

            record.chromatic_number =
                stoi(remove_quotes(fields[3]));

            record.edge_list =
                remove_quotes(fields[4]);

            record.adjacency_upper_triangular =
                remove_quotes(fields[5]);

            record.graph =
                build_graph_from_edge_list(
                    record.n_nodes,
                    record.edge_list
                );

            graphs.push_back(record);

        }
        catch (const exception& e) {

            cerr << "WARNING: Could not parse row: "
                 << e.what() << endl;
        }
    }

    file.close();

    return graphs;
}

// ------------------------------------------------------------
// Boost VF2 exact graph-isomorphism test
// ------------------------------------------------------------
bool boost_isomorphic(
    const Graph& g1,
    const Graph& g2
) {

    bool found = false;

    auto callback =
        [&](auto /*mapping1*/, auto /*mapping2*/) {

            found = true;

            // Stop immediately after first isomorphism
            return false;
        };

    boost::vf2_graph_iso(
        g1,
        g2,
        callback
    );

    return found;
}

// ------------------------------------------------------------
// Main
// ------------------------------------------------------------
int main(int argc, char* argv[]) {

    if (argc != 3) {

        cerr << "Usage:\n";
        cerr << "./isomorphism_check train.csv test.csv\n";

        return 1;
    }

    string train_file = argv[1];
    string test_file  = argv[2];

    cout << "====================================================\n";
    cout << "        BOOST VF2 GRAPH ISOMORPHISM CHECK\n";
    cout << "====================================================\n\n";

    // --------------------------------------------------------
    // Read datasets
    // --------------------------------------------------------

    cout << "Reading training graphs...\n";

    vector<GraphRecord> train_graphs =
        read_csv(train_file);

    cout << "Training graphs: "
         << train_graphs.size()
         << "\n\n";

    cout << "Reading test graphs...\n";

    vector<GraphRecord> test_graphs =
        read_csv(test_file);

    cout << "Test graphs: "
         << test_graphs.size()
         << "\n\n";

    // ========================================================
    // TEST vs TRAIN
    // ========================================================

    cout << "====================================================\n";
    cout << "TEST vs TRAIN\n";
    cout << "====================================================\n";

    string train_output =
        "test_vs_train_isomorphism_results.csv";

    ofstream train_csv(train_output);

    train_csv
        << "test_id,"
        << "train_id,"
        << "test_n_nodes,"
        << "train_n_nodes,"
        << "test_n_edges,"
        << "train_n_edges,"
        << "boost_checked,"
        << "isomorphic\n";

    long long total_train_comparisons = 0;
    long long actual_boost_train_checks = 0;
    long long train_isomorphic_pairs = 0;

    for (size_t i = 0; i < test_graphs.size(); ++i) {

        const auto& test = test_graphs[i];

        cout << "Test graph "
             << test.generated_id
             << " (" << i + 1
             << "/" << test_graphs.size()
             << ")";

        long long matches_for_test = 0;

        for (size_t j = 0;
             j < train_graphs.size();
             ++j) {

            const auto& train = train_graphs[j];

            total_train_comparisons++;

            bool iso = false;
            bool boost_checked = false;

            // ------------------------------------------------
            // Quick necessary-condition filtering
            //
            // Two isomorphic graphs MUST have:
            //   same number of vertices
            //   same number of edges
            //
            // This is not the actual isomorphism test.
            // ------------------------------------------------
            if (test.n_nodes == train.n_nodes &&
                test.n_edges == train.n_edges) {

                boost_checked = true;

                actual_boost_train_checks++;

                iso = boost_isomorphic(
                    test.graph,
                    train.graph
                );
            }

            if (iso) {

                train_isomorphic_pairs++;
                matches_for_test++;
            }

            train_csv
                << test.generated_id << ","
                << train.generated_id << ","
                << test.n_nodes << ","
                << train.n_nodes << ","
                << test.n_edges << ","
                << train.n_edges << ","
                << (boost_checked ? 1 : 0) << ","
                << (iso ? 1 : 0)
                << "\n";
        }

        cout << " -> isomorphic training graphs: "
             << matches_for_test
             << "\n";
    }

    train_csv.close();

    // ========================================================
    // TEST vs TEST
    // ========================================================

    cout << "\n";
    cout << "====================================================\n";
    cout << "TEST vs TEST\n";
    cout << "====================================================\n";

    string test_output =
        "test_vs_test_isomorphism_results.csv";

    ofstream test_csv(test_output);

    test_csv
        << "test_id_1,"
        << "test_id_2,"
        << "n_nodes_1,"
        << "n_nodes_2,"
        << "n_edges_1,"
        << "n_edges_2,"
        << "boost_checked,"
        << "isomorphic\n";

    long long total_test_comparisons = 0;
    long long actual_boost_test_checks = 0;
    long long test_isomorphic_pairs = 0;

    for (size_t i = 0;
         i < test_graphs.size();
         ++i) {

        for (size_t j = i + 1;
             j < test_graphs.size();
             ++j) {

            const auto& g1 = test_graphs[i];
            const auto& g2 = test_graphs[j];

            total_test_comparisons++;

            bool iso = false;
            bool boost_checked = false;

            // Necessary conditions
            if (g1.n_nodes == g2.n_nodes &&
                g1.n_edges == g2.n_edges) {

                boost_checked = true;

                actual_boost_test_checks++;

                iso = boost_isomorphic(
                    g1.graph,
                    g2.graph
                );
            }

            if (iso) {
                test_isomorphic_pairs++;
            }

            test_csv
                << g1.generated_id << ","
                << g2.generated_id << ","
                << g1.n_nodes << ","
                << g2.n_nodes << ","
                << g1.n_edges << ","
                << g2.n_edges << ","
                << (boost_checked ? 1 : 0) << ","
                << (iso ? 1 : 0)
                << "\n";
        }
    }

    test_csv.close();

    // ========================================================
    // FIND NOVEL TEST GRAPHS
    // ========================================================

    vector<long long> novel_test_ids;

    for (const auto& test : test_graphs) {

        bool exists_in_training = false;

        for (const auto& train : train_graphs) {

            // Only possible if these invariants match
            if (test.n_nodes != train.n_nodes ||
                test.n_edges != train.n_edges) {

                continue;
            }

            if (boost_isomorphic(
                    test.graph,
                    train.graph)) {

                exists_in_training = true;
                break;
            }
        }

        if (!exists_in_training) {

            novel_test_ids.push_back(
                test.generated_id
            );
        }
    }

    // ========================================================
    // SUMMARY
    // ========================================================

    string summary_output =
        "isomorphism_summary.csv";

    ofstream summary_csv(summary_output);

    summary_csv << "metric,value\n";

    summary_csv
        << "training_graphs,"
        << train_graphs.size()
        << "\n";

    summary_csv
        << "test_graphs,"
        << test_graphs.size()
        << "\n";

    summary_csv
        << "test_vs_train_total_comparisons,"
        << total_train_comparisons
        << "\n";

    summary_csv
        << "test_vs_train_actual_boost_checks,"
        << actual_boost_train_checks
        << "\n";

    summary_csv
        << "test_vs_train_isomorphic_pairs,"
        << train_isomorphic_pairs
        << "\n";

    summary_csv
        << "test_vs_test_total_comparisons,"
        << total_test_comparisons
        << "\n";

    summary_csv
        << "test_vs_test_actual_boost_checks,"
        << actual_boost_test_checks
        << "\n";

    summary_csv
        << "test_vs_test_isomorphic_pairs,"
        << test_isomorphic_pairs
        << "\n";

    summary_csv
        << "novel_test_graphs,"
        << novel_test_ids.size()
        << "\n";

    summary_csv.close();

    // ========================================================
    // PRINT FINAL RESULTS
    // ========================================================

    cout << "\n";
    cout << "====================================================\n";
    cout << "FINAL RESULTS\n";
    cout << "====================================================\n";

    cout << "\nTraining graphs              : "
         << train_graphs.size();

    cout << "\nTest graphs                  : "
         << test_graphs.size();

    cout << "\n";

    cout << "\nTEST vs TRAIN\n";

    cout << "Total possible comparisons  : "
         << total_train_comparisons;

    cout << "\nActual Boost checks         : "
         << actual_boost_train_checks;

    cout << "\nIsomorphic pairs             : "
         << train_isomorphic_pairs;

    cout << "\n";

    cout << "\nTEST vs TEST\n";

    cout << "Total possible comparisons  : "
         << total_test_comparisons;

    cout << "\nActual Boost checks         : "
         << actual_boost_test_checks;

    cout << "\nIsomorphic pairs             : "
         << test_isomorphic_pairs;

    cout << "\n";

    cout << "\nNovel test graphs            : "
         << novel_test_ids.size();

    cout << "\n";

    cout << "\nNovel test graph IDs:\n";

    for (long long id : novel_test_ids) {
        cout << id << " ";
    }

    cout << "\n";

    cout << "\n====================================================\n";
    cout << "OUTPUT FILES\n";
    cout << "====================================================\n";

    cout << "1. "
         << train_output
         << "\n";

    cout << "2. "
         << test_output
         << "\n";

    cout << "3. "
         << summary_output
         << "\n";

    cout << "\nDone.\n";

    return 0;
}

Writing isomorphism_check.cpp


In [3]:
!g++ -std=c++17 -O3 isomorphism_check.cpp \
    -o isomorphism_check \
    -lboost_graph

In [4]:
!./isomorphism_check \
    "Chromatic9_train(3).csv" \
    "generated_9_chromatic_graphs_test(10) (2).csv"

        BOOST VF2 GRAPH ISOMORPHISM CHECK

Reading training graphs...
Training graphs: 840

Reading test graphs...
Test graphs: 45

TEST vs TRAIN
Test graph 20 (1/45) -> isomorphic training graphs: 0
Test graph 22 (2/45) -> isomorphic training graphs: 0
Test graph 23 (3/45) -> isomorphic training graphs: 0
Test graph 24 (4/45) -> isomorphic training graphs: 0
Test graph 28 (5/45) -> isomorphic training graphs: 0
Test graph 29 (6/45) -> isomorphic training graphs: 0
Test graph 32 (7/45) -> isomorphic training graphs: 0
Test graph 49 (8/45) -> isomorphic training graphs: 0
Test graph 50 (9/45) -> isomorphic training graphs: 0
Test graph 52 (10/45) -> isomorphic training graphs: 0
Test graph 63 (11/45) -> isomorphic training graphs: 0
Test graph 83 (12/45) -> isomorphic training graphs: 0
Test graph 85 (13/45) -> isomorphic training graphs: 0
Test graph 90 (14/45) -> isomorphic training graphs: 0
Test graph 92 (15/45) -> isomorphic training graphs: 0
Test graph 104 (16/45) -> isomorphic t

In [5]:
import pandas as pd
from IPython.display import display

# ============================================================
# DISPLAY FINAL RESULTS IN GOOGLE COLAB
# ============================================================

# ------------------------------------------------------------
# 1. Read result files generated by the C++ program
# ------------------------------------------------------------

train_results = pd.read_csv(
    "test_vs_train_isomorphism_results.csv"
)

test_results = pd.read_csv(
    "test_vs_test_isomorphism_results.csv"
)

summary = pd.read_csv(
    "isomorphism_summary.csv"
)


# ============================================================
# 2. PRINT SUMMARY
# ============================================================

print("=" * 70)
print("              BOOST VF2 ISOMORPHISM RESULTS")
print("=" * 70)

for _, row in summary.iterrows():
    print(f"{row['metric']:45s}: {row['value']}")

print("=" * 70)


# ============================================================
# 3. TEST vs TRAIN RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("TEST vs TRAIN — ISOMORPHIC PAIRS")
print("=" * 70)

train_iso = train_results[
    train_results["isomorphic"] == 1
]

if len(train_iso) == 0:
    print("No test graph is isomorphic to any training graph.")
else:
    display(train_iso)


# ============================================================
# 4. TEST vs TEST RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("TEST vs TEST — ISOMORPHIC PAIRS")
print("=" * 70)

test_iso = test_results[
    test_results["isomorphic"] == 1
]

if len(test_iso) == 0:
    print("No pair of test graphs is isomorphic.")
else:
    display(test_iso)


# ============================================================
# 5. NOVEL TEST GRAPHS
# ============================================================

print("\n")
print("=" * 70)
print("NOVEL TEST GRAPHS")
print("=" * 70)

# Find test IDs that have NO isomorphic training graph
matched_test_ids = set(
    train_results.loc[
        train_results["isomorphic"] == 1,
        "test_id"
    ]
)

all_test_ids = set(
    train_results["test_id"].unique()
)

novel_test_ids = sorted(
    all_test_ids - matched_test_ids
)

print("Total test graphs :", len(all_test_ids))
print("Novel test graphs :", len(novel_test_ids))

print("\nNovel test graph IDs:")
print(novel_test_ids)


# ============================================================
# 6. COMPLETE TEST vs TRAIN TABLE
# ============================================================

print("\n")
print("=" * 70)
print("COMPLETE TEST vs TRAIN RESULTS")
print("=" * 70)

display(train_results)


# ============================================================
# 7. COMPLETE TEST vs TEST TABLE
# ============================================================

print("\n")
print("=" * 70)
print("COMPLETE TEST vs TEST RESULTS")
print("=" * 70)

display(test_results)


# ============================================================
# 8. SUMMARY AS A TABLE
# ============================================================

print("\n")
print("=" * 70)
print("SUMMARY TABLE")
print("=" * 70)

display(summary)

              BOOST VF2 ISOMORPHISM RESULTS
training_graphs                              : 840
test_graphs                                  : 45
test_vs_train_total_comparisons              : 37800
test_vs_train_actual_boost_checks            : 861
test_vs_train_isomorphic_pairs               : 0
test_vs_test_total_comparisons               : 990
test_vs_test_actual_boost_checks             : 26
test_vs_test_isomorphic_pairs                : 0
novel_test_graphs                            : 45


TEST vs TRAIN — ISOMORPHIC PAIRS
No test graph is isomorphic to any training graph.


TEST vs TEST — ISOMORPHIC PAIRS
No pair of test graphs is isomorphic.


NOVEL TEST GRAPHS
Total test graphs : 45
Novel test graphs : 45

Novel test graph IDs:
[np.int64(20), np.int64(22), np.int64(23), np.int64(24), np.int64(28), np.int64(29), np.int64(32), np.int64(49), np.int64(50), np.int64(52), np.int64(63), np.int64(83), np.int64(85), np.int64(90), np.int64(92), np.int64(104), np.int64(105), np.int64(110),

,test_id,train_id,test_n_nodes,train_n_nodes,test_n_edges,train_n_edges,boost_checked,isomorphic
0,20,1,15,12,71,56,0,0
1,20,2,15,12,71,56,0,0
2,20,3,15,12,71,56,0,0
3,20,4,15,12,71,56,0,0
4,20,5,15,12,71,56,0,0
...,...,...,...,...,...,...,...,...
37795,288,836,15,16,71,84,0,0
37796,288,837,15,16,71,84,0,0
37797,288,838,15,16,71,84,0,0
37798,288,839,15,16,71,84,0,0




COMPLETE TEST vs TEST RESULTS


,test_id_1,test_id_2,n_nodes_1,n_nodes_2,n_edges_1,n_edges_2,boost_checked,isomorphic
0,20,22,15,15,71,75,0,0
1,20,23,15,12,71,60,0,0
2,20,24,15,15,71,74,0,0
3,20,28,15,14,71,69,0,0
4,20,29,15,13,71,64,0,0
...,...,...,...,...,...,...,...,...
985,267,285,13,13,64,63,0,0
986,267,288,13,15,64,71,0,0
987,274,285,15,13,72,63,0,0
988,274,288,15,15,72,71,0,0




SUMMARY TABLE


,metric,value
0,training_graphs,840
1,test_graphs,45
2,test_vs_train_total_comparisons,37800
3,test_vs_train_actual_boost_checks,861
4,test_vs_train_isomorphic_pairs,0
5,test_vs_test_total_comparisons,990
6,test_vs_test_actual_boost_checks,26
7,test_vs_test_isomorphic_pairs,0
8,novel_test_graphs,45
